#  Feature Engineering & Preprocessing Pipeline - US Visa Approval Prediction

---

##  **1. Objectives of Feature Engineering**
Based on the key findings from our **Exploratory Data Analysis (EDA)**, this notebook implements all necessary transformations, encodings, and scaling pipelines required to prepare the dataset for machine learning models.

### **Key Tasks Covered:**
1. **Data Cleaning & Anomaly Correction:** Handle negative employee counts and drop arbitrary identifiers (`case_id`).
2. **Feature Extraction:** Derive `company_age` from `yr_of_estab`.
3. **Wage Standardization:** Convert `prevailing_wage` across different units (Hour, Week, Month, Year) into a unified **Annual Wage**.
4. **Target Encoding & Decoding:** Map `Certified` ➔ `1` and `Denied` ➔ `0`, with decoding utilities.
5. **Handling Skewness & Outliers:** Apply `PowerTransformer` (Yeo-Johnson) and `StandardScaler` to numerical features.
6. **Categorical Encoding Strategy:**
   * **Ordinal Encoding:** For `education_of_employee` (High School < Bachelor's < Master's < Doctorate).
   * **One-Hot Encoding (`drop='first'`):** For nominal features (`continent`, `region_of_employment`, `has_job_experience`, etc.).
7. **Building a Production `ColumnTransformer` Pipeline:** Exportable preprocessor for both training and real-time prediction.
8. **Class Imbalance Handling:** Apply **SMOTE** on training data to balance `Certified` vs `Denied` classes.
9. **Verification & Serialization:** Save `preprocessor.pkl` using `dill` and test sample inference.


##  **2. Import Libraries & Setup**


In [13]:
# Core Data Manipulation
import pandas as pd
import numpy as np
import datetime
import os
import dill
import warnings
warnings.filterwarnings('ignore')

# Scikit-Learn Preprocessing & Pipelines
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    OneHotEncoder, 
    OrdinalEncoder, 
    StandardScaler, 
    PowerTransformer
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Imbalanced Learning
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE

print("✅ All feature engineering libraries successfully loaded!")


✅ All feature engineering libraries successfully loaded!


##  **3. Load & Inspect Dataset**


In [14]:
# Load the dataset
df = pd.read_csv("EasyVisa.csv")

print(f"Dataset Shape: {df.shape}")
df.head()


Dataset Shape: (25480, 12)


,case_id,continent,education_of_employee,has_job_experience,requires_job_training,no_of_employees,yr_of_estab,region_of_employment,prevailing_wage,unit_of_wage,full_time_position,case_status
0,EZYV01,Asia,High School,N,N,14513,2007,West,592.2029,Hour,Y,Denied
1,EZYV02,Asia,Master's,Y,N,2412,2002,Northeast,83425.6500,Year,Y,Certified
2,EZYV03,Asia,Bachelor's,N,Y,44444,2008,West,122996.8600,Year,Y,Denied
3,EZYV04,Asia,Bachelor's,N,N,98,1897,West,83434.0300,Year,Y,Denied
4,EZYV05,Africa,Master's,Y,N,1082,2005,South,149907.3900,Year,Y,Certified


##  **4. Data Cleaning & Feature Extraction**


In [15]:
# 1. Clean Negative Values in `no_of_employees`
# EDA revealed negative counts (e.g. -26), which are data-entry errors. We take the absolute value.
df['no_of_employees'] = df['no_of_employees'].abs()
print(f"Minimum no_of_employees after cleaning: {df['no_of_employees'].min()}")

# 2. Derive Feature: `company_age`
current_year = datetime.date.today().year
df['company_age'] = current_year - df['yr_of_estab']
print(f"Company age range: {df['company_age'].min()} to {df['company_age'].max()} years")

# 3. Standardize `prevailing_wage` to Annual Salary
# Units: Hour (* 2080), Week (* 52), Month (* 12), Year (* 1)
def calculate_annual_wage(row):
    unit = row['unit_of_wage']
    wage = row['prevailing_wage']
    if unit == 'Hour':
        return wage * 2080   # 40 hrs/week * 52 weeks
    elif unit == 'Week':
        return wage * 52
    elif unit == 'Month':
        return wage * 12
    else:
        return wage

df['annual_prevailing_wage'] = df.apply(calculate_annual_wage, axis=1)

# 4. Drop redundant columns: `case_id`, `yr_of_estab`, `unit_of_wage`, `prevailing_wage`
df.drop(columns=['case_id', 'yr_of_estab', 'unit_of_wage', 'prevailing_wage'], inplace=True)

print("Updated DataFrame Shape after Feature Extraction:", df.shape)
df.head()


Minimum no_of_employees after cleaning: 11
Company age range: 10 to 226 years
Updated DataFrame Shape after Feature Extraction: (25480, 10)


,continent,education_of_employee,has_job_experience,requires_job_training,no_of_employees,region_of_employment,full_time_position,case_status,company_age,annual_prevailing_wage
0,Asia,High School,N,N,14513,West,Y,Denied,19,1231782.032
1,Asia,Master's,Y,N,2412,Northeast,Y,Certified,24,83425.650
2,Asia,Bachelor's,N,Y,44444,West,Y,Denied,18,122996.860
3,Asia,Bachelor's,N,N,98,West,Y,Denied,129,83434.030
4,Africa,Master's,Y,N,1082,South,Y,Certified,21,149907.390


###  **Feature Extraction Summary:**
* **`company_age`:** Derived from `current_year - yr_of_estab` (captures company maturity and stability).
* **`annual_prevailing_wage`:** Converted all wages to a standard annual rate, eliminating unit discrepancies.
* **`case_id` & `unit_of_wage` Dropped:** Removed redundant identifier and raw unit fields.


##  **5. Target Variable Encoding & Decoding**


In [16]:
# Target Mapping Dictionary
target_mapping = {'Certified': 1, 'Denied': 0}
target_decoding = {1: 'Certified', 0: 'Denied'}

# Apply mapping to target column
df['case_status'] = df['case_status'].map(target_mapping)

print("Target Class Distribution after Encoding:")
print(df['case_status'].value_counts())
print("Target Class Proportions:")
print(df['case_status'].value_counts(normalize=True).round(4) * 100)

# Function to decode predictions back to readable labels
def decode_prediction(pred_value):
    return target_decoding.get(pred_value, "Unknown")

print("Verification Decoding:")
print(f"Class 1 Decoded ➔ {decode_prediction(1)}")
print(f"Class 0 Decoded ➔ {decode_prediction(0)}")


Target Class Distribution after Encoding:
case_status
1    17018
0     8462
Name: count, dtype: int64
Target Class Proportions:
case_status
1    66.79
0    33.21
Name: proportion, dtype: float64
Verification Decoding:
Class 1 Decoded ➔ Certified
Class 0 Decoded ➔ Denied


##  **6. Feature Segregation & Stratified Train-Test Split**


In [17]:
# Separate Independent Features (X) and Dependent Target (y)
X = df.drop(columns=['case_status'])
y = df['case_status']

# Group Features by Transformation Type
ordinal_features = ['education_of_employee']

categorical_features = [
    'continent', 
    'has_job_experience', 
    'requires_job_training', 
    'region_of_employment', 
    'full_time_position'
]

numerical_features = ['no_of_employees', 'company_age', 'annual_prevailing_wage']

print(f"📌 Ordinal Features ({len(ordinal_features)}): {ordinal_features}")
print(f"📌 Nominal Categorical Features ({len(categorical_features)}): {categorical_features}")
print(f"📌 Numerical Features ({len(numerical_features)}): {numerical_features}")


📌 Ordinal Features (1): ['education_of_employee']
📌 Nominal Categorical Features (5): ['continent', 'has_job_experience', 'requires_job_training', 'region_of_employment', 'full_time_position']
📌 Numerical Features (3): ['no_of_employees', 'company_age', 'annual_prevailing_wage']


In [18]:
# Perform Stratified Train-Test Split (80% Train, 20% Test)
# Stratify=y ensures identical class proportions (66.8% Certified / 33.2% Denied) in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Set:   X_train = {X_train.shape}, y_train = {y_train.shape}")
print(f"Testing Set:    X_test  = {X_test.shape}, y_test  = {y_test.shape}")


Training Set:   X_train = (20384, 9), y_train = (20384,)
Testing Set:    X_test  = (5096, 9), y_test  = (5096,)


##  **7. Constructing the Preprocessor Pipeline (`ColumnTransformer`)**


In [19]:
# 1. Define Ordinal Categories in natural ranking order
education_order = ['High School', "Bachelor's", "Master's", 'Doctorate']

# 2. Sub-Pipelines for each data type
# Ordinal Pipeline: OrdinalEncoder + Standard Scaling
ordinal_pipeline = Pipeline(steps=[
    ('ordinal_encoder', OrdinalEncoder(categories=[education_order])),
    ('scaler', StandardScaler())
])

# Categorical Pipeline: OneHotEncoder (drop='first' to prevent dummy variable trap)
categorical_pipeline = Pipeline(steps=[
    ('one_hot_encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')),
    ('scaler', StandardScaler())
])

# Numerical Pipeline: PowerTransformer (Yeo-Johnson to reduce skewness) + StandardScaler
numerical_pipeline = Pipeline(steps=[
    ('power_transformer', PowerTransformer(method='yeo-johnson')),
    ('scaler', StandardScaler())
])

# 3. Combine all sub-pipelines into a Single Master ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('ordinal_pipeline', ordinal_pipeline, ordinal_features),
        ('categorical_pipeline', categorical_pipeline, categorical_features),
        ('numerical_pipeline', numerical_pipeline, numerical_features)
    ],
    remainder='drop'
)

print("✅ Master Preprocessor Pipeline built successfully!")
preprocessor


✅ Master Preprocessor Pipeline built successfully!


ColumnTransformer(transformers=[('ordinal_pipeline',
                                 Pipeline(steps=[('ordinal_encoder',
                                                  OrdinalEncoder(categories=[['High '
                                                                              'School',
                                                                              "Bachelor's",
                                                                              "Master's",
                                                                              'Doctorate']])),
                                                 ('scaler', StandardScaler())]),
                                 ['education_of_employee']),
                                ('categorical_pipeline',
                                 Pipeline(steps=[('one_hot_encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False)),
                                                 ('scaler', StandardScaler())]),
                                 ['continent', 'has_job_experience',
                                  'requires_job_training',
                                  'region_of_employment',
                                  'full_time_position']),
                                ('numerical_pipeline',
                                 Pipeline(steps=[('power_transformer',
                                                  PowerTransformer()),
                                                 ('scaler', StandardScaler())]),
                                 ['no_of_employees', 'company_age',
                                  'annual_prevailing_wage'])])

##  **8. Fitting Preprocessor on Training Data & Transforming Test Data**


In [20]:
# Fit on Training Data ONLY (Prevents Data Leakage) and transform
X_train_transformed = preprocessor.fit_transform(X_train)

# Transform Test Data using the fitted parameters
X_test_transformed = preprocessor.transform(X_test)

print("="*60)
print(f"Transformed X_train Shape: {X_train_transformed.shape}")
print(f"Transformed X_test Shape:  {X_test_transformed.shape}")
print("="*60)


Transformed X_train Shape: (20384, 16)
Transformed X_test Shape:  (5096, 16)


In [21]:
# Extract Generated Feature Names for Transparency
encoded_cat_names = preprocessor.named_transformers_['categorical_pipeline']['one_hot_encoder'].get_feature_names_out(categorical_features).tolist()
all_transformed_feature_names = ordinal_features + encoded_cat_names + numerical_features

print(f"Total Transformed Features ({len(all_transformed_feature_names)}):")
for i, name in enumerate(all_transformed_feature_names):
    print(f"  {i+1:02d}. {name}")

# Display sample transformed training dataframe
pd.DataFrame(X_train_transformed, columns=all_transformed_feature_names).head()


Total Transformed Features (16):
  01. education_of_employee
  02. continent_Asia
  03. continent_Europe
  04. continent_North America
  05. continent_Oceania
  06. continent_South America
  07. has_job_experience_Y
  08. requires_job_training_Y
  09. region_of_employment_Midwest
  10. region_of_employment_Northeast
  11. region_of_employment_South
  12. region_of_employment_West
  13. full_time_position_Y
  14. no_of_employees
  15. company_age
  16. annual_prevailing_wage


,education_of_employee,continent_Asia,continent_Europe,continent_North America,continent_Oceania,continent_South America,has_job_experience_Y,requires_job_training_Y,region_of_employment_Midwest,region_of_employment_Northeast,region_of_employment_South,region_of_employment_West,full_time_position_Y,no_of_employees,company_age,annual_prevailing_wage
0,-0.502741,0.714639,-0.413305,-0.385304,-0.09006,-0.185771,0.852225,-0.36081,-0.450579,-0.625999,1.616250,-0.590343,0.348095,0.630852,0.461805,-0.215701
1,-0.502741,0.714639,-0.413305,-0.385304,-0.09006,-0.185771,0.852225,-0.36081,2.219365,-0.625999,-0.618716,-0.590343,0.348095,0.394926,1.010417,-0.651883
2,-0.502741,-1.399307,-0.413305,2.595354,-0.09006,-0.185771,-1.173399,-0.36081,-0.450579,-0.625999,1.616250,-0.590343,0.348095,-0.717242,0.816978,0.818976
3,0.705164,0.714639,-0.413305,-0.385304,-0.09006,-0.185771,0.852225,-0.36081,-0.450579,1.597447,-0.618716,-0.590343,0.348095,-0.401682,0.880555,-0.449917
4,-0.502741,-1.399307,-0.413305,2.595354,-0.09006,-0.185771,0.852225,-0.36081,-0.450579,-0.625999,-0.618716,1.693930,0.348095,-0.670502,0.616770,0.103219


##  **9. Handling Class Imbalance with SMOTE**


In [22]:
# Class distribution before SMOTE
print("="*60)
print("📌 CLASS DISTRIBUTION BEFORE SMOTE:")
print("="*60)
print(y_train.value_counts())
print(f"Certified (1): {sum(y_train == 1):,} | Denied (0): {sum(y_train == 0):,}")

# Apply SMOTE (Synthetic Minority Over-sampling Technique)
# Apply ONLY on X_train_transformed (never on test data to avoid evaluation bias)
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_transformed, y_train)

print("" + "="*60)
print("📌 CLASS DISTRIBUTION AFTER SMOTE:")
print("="*60)
print(y_train_resampled.value_counts())
print(f"Certified (1): {sum(y_train_resampled == 1):,} | Denied (0): {sum(y_train_resampled == 0):,}")
print(f"Resampled Training Matrix Shape: {X_train_resampled.shape}")


📌 CLASS DISTRIBUTION BEFORE SMOTE:
case_status
1    13614
0     6770
Name: count, dtype: int64
Certified (1): 13,614 | Denied (0): 6,770
📌 CLASS DISTRIBUTION AFTER SMOTE:
case_status
1    13614
0    13614
Name: count, dtype: int64
Certified (1): 13,614 | Denied (0): 13,614
Resampled Training Matrix Shape: (27228, 16)


##  **10. Serialize Preprocessor Object for Production Pipeline**


In [23]:
# Create artifact directory if it doesn't exist
os.makedirs("models", exist_ok=True)
preprocessor_path = os.path.join("models", "preprocessor.pkl")

# Save preprocessor using dill
with open(preprocessor_path, "wb") as f:
    dill.dump(preprocessor, f)

print(f"✅ Preprocessor successfully serialized and saved to: {preprocessor_path}")

# Test Loading Preprocessor
with open(preprocessor_path, "rb") as f:
    loaded_preprocessor = dill.load(f)

# Test Inference on a Sample Single Applicant Data Dictionary
sample_applicant = pd.DataFrame([{
    'continent': 'Asia',
    'education_of_employee': "Master's",
    'has_job_experience': 'Y',
    'requires_job_training': 'N',
    'no_of_employees': 1500,
    'region_of_employment': 'West',
    'full_time_position': 'Y',
    'company_age': 18,
    'annual_prevailing_wage': 95000.0
}])

sample_transformed = loaded_preprocessor.transform(sample_applicant)
print("Sample Single Applicant Input:")
print(sample_applicant.to_dict(orient='records')[0])
print(f"Transformed Vector Shape: {sample_transformed.shape}")
print("Transformed Feature Vector:")
print(sample_transformed)


✅ Preprocessor successfully serialized and saved to: models\preprocessor.pkl
Sample Single Applicant Input:
{'continent': 'Asia', 'education_of_employee': "Master's", 'has_job_experience': 'Y', 'requires_job_training': 'N', 'no_of_employees': 1500, 'region_of_employment': 'West', 'full_time_position': 'Y', 'company_age': 18, 'annual_prevailing_wage': 95000.0}
Transformed Vector Shape: (1, 16)
Transformed Feature Vector:
[[ 0.70516428  0.71463925 -0.41330542 -0.38530386 -0.09005988 -0.18577072
   0.85222489 -0.36081031 -0.45057931 -0.62599869 -0.61871629  1.69393039
   0.34809546 -0.16299455 -1.08549964  0.15296516]]


##  **11. Feature Engineering Summary & Model Training Readiness**

---

###  **Transformation Summary Table**

| Feature | Data Type | Engineering / Transformation Applied | Output Representation |
| :--- | :--- | :--- | :--- |
| **`education_of_employee`** | Categorical | Ordinal Encoding (`High School` < `Bachelor's` < `Master's` < `Doctorate`) + Scaled | 1 Scaled Numerical Column |
| **`continent`** | Categorical | One-Hot Encoding (`drop='first'`) + Scaled | 5 Dummy Columns |
| **`has_job_experience`** | Categorical | One-Hot Encoding (`drop='first'`) + Scaled | 1 Dummy Column (`Y` ➔ 1) |
| **`requires_job_training`** | Categorical | One-Hot Encoding (`drop='first'`) + Scaled | 1 Dummy Column (`Y` ➔ 1) |
| **`region_of_employment`** | Categorical | One-Hot Encoding (`drop='first'`) + Scaled | 4 Dummy Columns |
| **`full_time_position`** | Categorical | One-Hot Encoding (`drop='first'`) + Scaled | 1 Dummy Column (`Y` ➔ 1) |
| **`no_of_employees`** | Numerical | Cleaned absolute values + `PowerTransformer(Yeo-Johnson)` + `StandardScaler` | 1 Normalized Column |
| **`company_age`** | Numerical | Derived (`2026 - yr_of_estab`) + `PowerTransformer` + `StandardScaler` | 1 Normalized Column |
| **`annual_prevailing_wage`**| Numerical | Converted to annual basis + `PowerTransformer` + `StandardScaler` | 1 Normalized Column |
| **`case_status` (Target)** | Target | Mapped (`Certified` ➔ 1, `Denied` ➔ 0) | Binary Integer Vector |

---

###  **Ready for Model Training:**
* Transformed and Balanced training data: `X_train_resampled`, `y_train_resampled`
* Transformed evaluation test data: `X_test_transformed`, `y_test`
* Serialized preprocessing object: `models/preprocessor.pkl`

👉 **Next Step:** Model Training & Hyperparameter Tuning (`model_training.ipynb`)!
